In [1]:
import geopandas as gpd
import pandas as pd
stores = pd.read_csv("bunnings geolocations.csv")
sa2=gpd.read_file(r'D:\SA2_2021_AUST_SHP_GDA2020\SA2_2021_AUST_GDA2020.shp')
stores_gdf = gpd.GeoDataFrame(
    stores,
    # This turns each (lon, lat) pair into a geometric point object.
    geometry=gpd.points_from_xy(stores['longitude'], stores['latitude']),
    # crs=EPSG:7844 is coordinate refernce system. its code for GDA202, australia official datum
    crs='EPSG:7844'
#converts your store points to same as SA2shapefile
).to_crs(sa2.crs)
#predicate='within' its spatial relationship to test it asks is this point geometrically inside this polygon?
joined = gpd.sjoin(stores_gdf, sa2, how='left', predicate='within')
result = joined[['store_name', 'latitude', 'longitude',
                 'SA2_CODE21', 'SA2_NAME21', 'SA3_NAME21', 'SA4_NAME21', 'STE_NAME21']].copy()
result.columns = ['store_name', 'latitude', 'longitude',
                  'sa2_code', 'sa2_name', 'sa3_name', 'sa4_name', 'state']
result.to_csv(r'D:\full AUS\bunnings_complete_with_sa2.csv', index=False)

print(f"  Stores matched to an SA2: {result['sa2_code'].notna().sum()}")
print(f"  Stores with no SA2:        {result['sa2_code'].isna().sum()}")
print(f"\n>>> UNIQUE POSITIVE SA2s: {result['sa2_code'].nunique()} <<<")
print(f"\nState breakdown:")
print(result['state'].value_counts())

  Stores matched to an SA2: 312
  Stores with no SA2:        0

>>> UNIQUE POSITIVE SA2s: 301 <<<

State breakdown:
state
New South Wales                 94
Victoria                        79
Queensland                      64
Western Australia               38
South Australia                 22
Tasmania                         7
Australian Capital Territory     5
Northern Territory               3
Name: count, dtype: int64


**Note:** This section requires the ABS shapefile and Census GCP files, which aren't included in this repo (too large). Download them from links in `data/README.md` and update the paths below to your local copy. The processed outputs are saved as CSVs in this repo, so all subsequent sections run without re-running this part.

In [2]:
df = pd.read_csv('bunnings_complete_with_sa2.csv')

# Show the three suspect rows with their full coordinates
suspect_names = ['Caroline Springs', 'Sunshine', 'Carrum Downs', 'Dandenong', 'Gympie', 'Lawnton']
print(df[df['store_name'].isin(suspect_names)][
    ['store_name', 'latitude', 'longitude', 'sa2_name']
].to_string())

           store_name   latitude   longitude        sa2_name
121            Gympie -27.287964  152.985367         Lawnton
128           Lawnton -27.287964  152.985367         Lawnton
203  Caroline Springs -37.775852  144.830221  Sunshine North
204      Carrum Downs -38.095511  145.174008    Carrum Downs
213         Dandenong -38.095511  145.174008    Carrum Downs
258          Sunshine -37.775852  144.830221  Sunshine North


In [3]:
# now we got three locations with wrong coordinates
import pandas as pd

# Load the current file
df = pd.read_csv("bunnings_complete_with_sa2.csv")

# Your three verified corrections
corrections = {
    'Gympie':           (-26.223300, 152.691310),
    'Caroline Springs': (-37.760320, 144.744730),
    'Dandenong':        (-38.005015, 145.250583),
}

# Apply each correction
for name, (lat, lon) in corrections.items():
    mask = df['store_name'] == name
    if mask.sum() == 1:
        old_lat = df.loc[mask, 'latitude'].values[0]
        old_lon = df.loc[mask, 'longitude'].values[0]
        df.loc[mask, 'latitude'] = lat
        df.loc[mask, 'longitude'] = lon
        print(f"{name}:")
        print(f"  Before: ({old_lat:.6f}, {old_lon:.6f})")
        print(f"  After:  ({lat:.6f}, {lon:.6f})")
    else:
        print(f"{name}: matched {mask.sum()} rows — check store name spelling")

# Save back to the same file (overwrites old version)
df.to_csv("bunnings_complete_with_sa2.csv", index=False)
print(f"\n File updated: D:\\full AUS\\bunnings_exact_complete.csv")
print(f"  Total stores: {len(df)}")

Gympie:
  Before: (-27.287964, 152.985367)
  After:  (-26.223300, 152.691310)
Caroline Springs:
  Before: (-37.775852, 144.830221)
  After:  (-37.760320, 144.744730)
Dandenong:
  Before: (-38.095511, 145.174008)
  After:  (-38.005015, 145.250583)

 File updated: D:\full AUS\bunnings_exact_complete.csv
  Total stores: 312


In [4]:
# no we rerun the spatial join
# Reload the corrected store file
stores = pd.read_csv("bunnings_complete_with_sa2.csv")

# Reload SA2 polygons
sa2 = gpd.read_file(r'D:\SA2_2021_AUST_SHP_GDA2020\SA2_2021_AUST_GDA2020.shp')

# Convert stores to geographic points
stores_gdf = gpd.GeoDataFrame(
    stores,
    geometry=gpd.points_from_xy(stores['longitude'], stores['latitude']),
    crs='EPSG:7844'
).to_crs(sa2.crs)

# Spatial join
joined = gpd.sjoin(stores_gdf, sa2, how='left', predicate='within')

# Clean output
result = joined[['store_name', 'latitude', 'longitude',
                 'SA2_CODE21', 'SA2_NAME21', 'SA3_NAME21', 'SA4_NAME21', 'STE_NAME21']].copy()
result.columns = ['store_name', 'latitude', 'longitude',
                  'sa2_code', 'sa2_name', 'sa3_name', 'sa4_name', 'state']

result.to_csv('bunnings_complete_with_sa2.csv', index=False)

# Show the three corrected stores' new SA2 assignments
print("Corrected stores and their new SA2s:")
print(result[result['store_name'].isin(['Gympie', 'Caroline Springs', 'Dandenong'])][
    ['store_name', 'sa2_name', 'sa3_name', 'state']
].to_string())

print(f"\n>>> NEW UNIQUE POSITIVE SA2s: {result['sa2_code'].nunique()} <<<")
print(f"\nStores matched: {result['sa2_code'].notna().sum()} / {len(result)}")
print(f"\nState breakdown:")
print(result['state'].value_counts())

Corrected stores and their new SA2s:
           store_name                   sa2_name                sa3_name       state
121            Gympie             Gympie - North       Gympie - Cooloola  Queensland
203  Caroline Springs  Rockbank - Mount Cottrell  Melton - Bacchus Marsh    Victoria
213         Dandenong          Dandenong - South               Dandenong    Victoria

>>> NEW UNIQUE POSITIVE SA2s: 304 <<<

Stores matched: 312 / 312

State breakdown:
state
New South Wales                 94
Victoria                        79
Queensland                      64
Western Australia               38
South Australia                 22
Tasmania                         7
Australian Capital Territory     5
Northern Territory               3
Name: count, dtype: int64


In [5]:
gcp_path = r'D:\full AUS\2021_GCP_SA2_for_AUS_short-header\2021 Census GCP Statistical Area 2 for AUS'
g01 = pd.read_csv(f"{gcp_path}\\2021Census_G01_AUST_SA2.csv")

print(f"G01 shape: {g01.shape}")
print(f"\nFirst 10 columns: {list(g01.columns[:10])}")
print(f"\nTotal columns: {len(g01.columns)}")
print(f"\nFirst row:")
print(g01.iloc[0])

G01 shape: (2472, 109)

First 10 columns: ['SA2_CODE_2021', 'Tot_P_M', 'Tot_P_F', 'Tot_P_P', 'Age_0_4_yr_M', 'Age_0_4_yr_F', 'Age_0_4_yr_P', 'Age_5_14_yr_M', 'Age_5_14_yr_F', 'Age_5_14_yr_P']

Total columns: 109

First row:
SA2_CODE_2021                 101021007
Tot_P_M                            2234
Tot_P_F                            2117
Tot_P_P                            4343
Age_0_4_yr_M                        111
                                ...    
Count_psns_occ_priv_dwgs_F         1902
Count_psns_occ_priv_dwgs_P         3878
Count_Persons_other_dwgs_M          288
Count_Persons_other_dwgs_F          251
Count_Persons_other_dwgs_P          535
Name: 0, Length: 109, dtype: int64


In [6]:
# from these columns we need only SA2_CODE_2021 and Tot_P_P 
gcp_path = r'D:\full AUS\2021_GCP_SA2_for_AUS_short-header\2021 Census GCP Statistical Area 2 for AUS'

# Load only the columns we need
g01 = pd.read_csv(
    f"{gcp_path}\\2021Census_G01_AUST_SA2.csv",
    usecols=['SA2_CODE_2021', 'Tot_P_P']
)

# Rename to friendly column names
g01 = g01.rename(columns={
    'SA2_CODE_2021': 'sa2_code',
    'Tot_P_P': 'population'
})

# Sanity checks
print(f"Rows loaded: {len(g01)}")
print(f"Columns: {list(g01.columns)}")
print(f"Dtypes:\n{g01.dtypes}\n")

# Check for any zero or negative populations (would indicate non-residential SA2s)
zeros = (g01['population'] == 0).sum()
print(f"\nSA2s with zero population: {zeros}")

# Show top 5 most populous SA2s — sanity check (should be big urban areas)
print("\nTop 5 most populous SA2s:")
print(g01.nlargest(5, 'population'))

Rows loaded: 2472
Columns: ['sa2_code', 'population']
Dtypes:
sa2_code      int64
population    int64
dtype: object


SA2s with zero population: 40

Top 5 most populous SA2s:
       sa2_code  population
1091  213051583       28116
742   205031093       26792
989   212011552       26783
1825  404031107       26480
2098  507051313       26046


In [7]:
# Cross-check top populous SA2s against the SA2 shapefile names
sa2 = gpd.read_file(r'D:\SA2_2021_AUST_SHP_GDA2020\SA2_2021_AUST_GDA2020.shp')
top_codes = [213051583, 205031093, 212011552, 404031107, 507051313]
for code in top_codes:
    row = sa2[sa2['SA2_CODE21'] == str(code)]
    if len(row):
        print(f"  {code}: {row.iloc[0]['SA2_NAME21']}, {row.iloc[0]['STE_NAME21']}")
    else:
        print(f"  {code}: NOT FOUND in shapefile")

  213051583: Tarneit - Central, Victoria
  205031093: Wonthaggi - Inverloch, Victoria
  212011552: Pakenham - South West, Victoria
  404031107: Plympton, South Australia
  507051313: Baldivis - South, Western Australia


In [8]:
# now we need to use g02 file for median_mortgage_repay_monthly 
g02 = pd.read_csv(
    f"{gcp_path}\\2021Census_G02_AUST_SA2.csv"
)

print(f"G02 shape: {g02.shape}")
print(f"\nAll columns:")
print(list(g02.columns))
print(f"\nFirst row:")
print(g02.iloc[0])

G02 shape: (2472, 9)

All columns:
['SA2_CODE_2021', 'Median_age_persons', 'Median_mortgage_repay_monthly', 'Median_tot_prsnl_inc_weekly', 'Median_rent_weekly', 'Median_tot_fam_inc_weekly', 'Average_num_psns_per_bedroom', 'Median_tot_hhd_inc_weekly', 'Average_household_size']

First row:
SA2_CODE_2021                    101021007.0
Median_age_persons                      51.0
Median_mortgage_repay_monthly         1732.0
Median_tot_prsnl_inc_weekly            760.0
Median_rent_weekly                     330.0
Median_tot_fam_inc_weekly             1886.0
Average_num_psns_per_bedroom             0.8
Median_tot_hhd_inc_weekly             1429.0
Average_household_size                   2.2
Name: 0, dtype: float64


In [9]:
# we need to extract that mortgage column and the values are float and above population values are int we cant merge both unsless we have same dattype
gcp_path = r'D:\full AUS\2021_GCP_SA2_for_AUS_short-header\2021 Census GCP Statistical Area 2 for AUS'

# Load only what we need
g02 = pd.read_csv(
    f"{gcp_path}\\2021Census_G02_AUST_SA2.csv",
    usecols=['SA2_CODE_2021', 'Median_mortgage_repay_monthly']
)

# Rename and fix sa2_code dtype (float → int) for consistency with G01
g02 = g02.rename(columns={
    'SA2_CODE_2021': 'sa2_code',
    'Median_mortgage_repay_monthly': 'median_mortgage_monthly'
})
g02['sa2_code'] = g02['sa2_code'].astype('int64')

print(f"Rows loaded: {len(g02)}")
print(f"Dtypes:\n{g02.dtypes}\n")
#how many missing values
missing=g02['median_mortgage_monthly'].isna().sum()
zeros=(g02['median_mortgage_monthly']==0).sum()
print(f'missing:{missing}')
print(f'zeros:{zeros}')
# valid mortgagae values excluding zeros and missing values
valid=g02[g02['median_mortgage_monthly']>0]
print(valid.describe())

Rows loaded: 2472
Dtypes:
sa2_code                   int64
median_mortgage_monthly    int64
dtype: object

missing:0
zeros:111
           sa2_code  median_mortgage_monthly
count  2.361000e+03              2361.000000
mean   3.087233e+08              1883.896654
std    1.907041e+08               589.772546
min    1.010210e+08               109.000000
25%    1.270117e+08              1504.000000
50%    3.020210e+08              1850.000000
75%    4.040111e+08              2167.000000
max    9.010410e+08              9999.000000


In [10]:
# How many SA2s have the suspicious 9999 value?
sentinel_count = (g02['median_mortgage_monthly'] == 9999).sum()
print(f"SA2s with mortgage = 9999 (likely sentinel): {sentinel_count}")

# Show the distribution near the top to see if 9999 is a cluster
print("\nTop 15 mortgage values (look for repeated 9999s):")
top = g02.nlargest(15, 'median_mortgage_monthly')[['sa2_code', 'median_mortgage_monthly']]
print(top.to_string())

# Also show the lower end — 111 zeros, but are there other sentinels?
print("\nLowest 10 non-zero mortgage values:")
nonzero = g02[g02['median_mortgage_monthly'] > 0].nsmallest(10, 'median_mortgage_monthly')
print(nonzero.to_string())

SA2s with mortgage = 9999 (likely sentinel): 1

Top 15 mortgage values (look for repeated 9999s):
       sa2_code  median_mortgage_monthly
1931  503021037                     9999
1330  306031162                     5200
1525  312021345                     4800
456   121011683                     4333
582   126021499                     4333
2463  801111141                     4333
376   118011346                     4195
377   118011347                     4000
390   118021654                     4000
437   120021387                     4000
478   121041689                     4000
479   122011418                     4000
1924  503011030                     4000
1926  503011032                     3925
631   128011605                     3900

Lowest 10 non-zero mortgage values:
       sa2_code  median_mortgage_monthly
2314  702021055                      109
2313  702011054                      212
2320  702031061                      303
1857  406021139                      630
2147

In [11]:
# now we found 9999 that abnormal value we need to clean and replace that na dzero values to NAn
import numpy as np
g02.loc[g02['median_mortgage_monthly'].isin([0, 9999]), 'median_mortgage_monthly'] = np.nan
print(f"Dtypes after cleanup:")
print(g02.dtypes)
print()
print(f"\nClean mortgage distribution:")
print(g02['median_mortgage_monthly'].describe())

Dtypes after cleanup:
sa2_code                     int64
median_mortgage_monthly    float64
dtype: object


Clean mortgage distribution:
count    2360.000000
mean     1880.458051
std       565.730349
min       109.000000
25%      1503.000000
50%      1850.000000
75%      2167.000000
max      5200.000000
Name: median_mortgage_monthly, dtype: float64


In [12]:
import numpy as np
# now we need to add more columns like contruction_pct and retail_pct
gcp_path = r'D:\full AUS\2021_GCP_SA2_for_AUS_short-header\2021 Census GCP Statistical Area 2 for AUS'
g56a = pd.read_csv(
    f"{gcp_path}\\2021Census_G56A_AUST_SA2.csv",
    usecols=['SA2_CODE_2021', 'Construction_Tot', 'RetTde_Tot']
)
g56b = pd.read_csv(
    f"{gcp_path}\\2021Census_G56B_AUST_SA2.csv",
    usecols=['SA2_CODE_2021', 'Tot_Tot']
)
# now merge both files with we wanted columns
g56=g56a.merge(g56b, on='SA2_CODE_2021', how ='inner')
g56=g56.rename(columns={
    'SA2_CODE_2021': 'sa2_code',
    'Construction_Tot': 'construction_workers',
    'RetTde_Tot': 'retail_workers',
    'Tot_Tot': 'total_workers'
})
g56['sa2_code'] = g56['sa2_code'].astype('int64')
#check whether there are zero workers it causes infinity while dividing
zero_workers = (g56['total_workers'] == 0).sum()
print(f"SA2s with zero total workers: {zero_workers}")
# avoids zerodivisionerror
g56['construction_pct']=np.where( g56['total_workers']>0,
                                 (g56['construction_workers']/g56['total_workers'])*100,
                                 np.nan
                                )
g56['retail_pct'] = np.where(
    g56['total_workers'] > 0,
    (g56['retail_workers'] / g56['total_workers']) * 100,
    np.nan
)
# just keeping columns we wanted for megre
g56_clean=g56[['sa2_code','construction_pct','retail_pct']]
print(g56_clean.shape)

SA2s with zero total workers: 57
(2472, 3)


In [13]:
#sanity check
print("Construction % stats:")
print(g56['construction_pct'].describe())
print("\nRetail % stats:")
print(g56['retail_pct'].describe())
print("\nTop 5 highest construction % SA2s:")
print(g56.nlargest(5, 'construction_pct')[['sa2_code', 'construction_pct', 'total_workers']])

Construction % stats:
count    2415.000000
mean        8.755529
std         4.016666
min         0.000000
25%         6.637837
50%         8.558406
75%        10.600576
max       100.000000
Name: construction_pct, dtype: float64

Retail % stats:
count    2415.000000
mean        8.760170
std         3.946431
min         0.000000
25%         7.458625
50%         8.898509
75%        10.214010
max       133.333333
Name: retail_pct, dtype: float64

Top 5 highest construction % SA2s:
       sa2_code  construction_pct  total_workers
1473  310041298        100.000000              4
593   127011592         54.545455             11
1204  302031036         50.000000              8
1384  308051532         44.444444              9
1958  504031063         30.769231             13


In [14]:
#we got issue at max reatil_pct and with less total_workers we need to change that by assuming workers>100
gcp_path = r'D:\full AUS\2021_GCP_SA2_for_AUS_short-header\2021 Census GCP Statistical Area 2 for AUS'
# Reload sources (clean slate)
g56a = pd.read_csv(
    f"{gcp_path}\\2021Census_G56A_AUST_SA2.csv",
    usecols=['SA2_CODE_2021', 'Construction_Tot', 'RetTde_Tot']
).rename(columns={
    'SA2_CODE_2021': 'sa2_code',
    'Construction_Tot': 'construction_workers',
    'RetTde_Tot': 'retail_workers'
})

g56b = pd.read_csv(
    f"{gcp_path}\\2021Census_G56B_AUST_SA2.csv",
    usecols=['SA2_CODE_2021', 'Tot_Tot']
).rename(columns={'SA2_CODE_2021': 'sa2_code', 'Tot_Tot': 'total_workers'})
g56 = g56a.merge(g56b, on='sa2_code')

WORKER_THRESHOLD = 100

# Step 1: compute raw percentages (replace 0 in denominator with NaN to avoid div-by-zero warnings)
g56['construction_pct'] = (g56['construction_workers'] / g56['total_workers'].replace(0, np.nan)) * 100
g56['retail_pct']       = (g56['retail_workers']       / g56['total_workers'].replace(0, np.nan)) * 100

# Step 2: identify invalid rows (too few workers OR percentage > 100)
too_few_workers = g56['total_workers'] < WORKER_THRESHOLD
above_100_construction = g56['construction_pct'] > 100
above_100_retail = g56['retail_pct'] > 100

invalid_rows = too_few_workers | above_100_construction | above_100_retail

# Step 3: set the percentage to NaN for invalid rows
g56.loc[invalid_rows, 'construction_pct'] = np.nan
g56.loc[invalid_rows, 'retail_pct'] = np.nan



In [15]:
# Report
print(invalid_rows.sum())
print(f"Valid construction_pct: {g56['construction_pct'].notna().sum()}")
print(f"Valid retail_pct:       {g56['retail_pct'].notna().sum()}")

print(f"\nConstruction % stats:")
print(g56['construction_pct'].describe())
print(f"\nRetail % stats:")
print(g56['retail_pct'].describe())

print(f"\nTop 5 construction %:")
print(g56.nlargest(5, 'construction_pct')[['sa2_code','construction_pct','total_workers']])

g56_clean = g56[['sa2_code', 'construction_pct', 'retail_pct']]
print(f"\n g56_clean shape: {g56_clean.shape}")

126
Valid construction_pct: 2346
Valid retail_pct:       2346

Construction % stats:
count    2346.000000
mean        8.809654
std         2.940632
min         0.000000
25%         6.751771
50%         8.616123
75%        10.631846
max        21.192958
Name: construction_pct, dtype: float64

Retail % stats:
count    2346.000000
mean        8.878065
std         1.934059
min         0.000000
25%         7.595719
50%         8.985423
75%        10.237018
max        15.441413
Name: retail_pct, dtype: float64

Top 5 construction %:
       sa2_code  construction_pct  total_workers
315   115041301         21.192958           5851
1004  212031308         20.608843           4139
880   209031211         19.758454           2070
881   209031212         19.519520           2664
612   127021518         19.493807           1857

 g56_clean shape: (2472, 3)


In [16]:
# cobining all four columns we extracted
features=g01.merge(g02,on='sa2_code',how='outer')\
            .merge(g56_clean,on='sa2_code',how='outer')
print(f"Merged feature matrix shape: {features.shape}")
print(f"Columns: {list(features.columns)}\n")
output_path = 'national_features_v1.csv'

Merged feature matrix shape: (2472, 5)
Columns: ['sa2_code', 'population', 'median_mortgage_monthly', 'construction_pct', 'retail_pct']



**Note:** The next cells read the raw ABS Time Series Profile CSVs to compute population growth. These files aren't included in this repo (too large; download from [ABS DataPacks](https://www.abs.gov.au/census/find-census-data/datapacks)). Update the path below to your local copy if you want to re-run this step. Otherwise, the computed output is already in `national_features_v3.csv`, and all later cells run without this section.

In [17]:
# Inspect T01 (Time Series Profile — selected person characteristics)
tsp_path = r'D:\full AUS\Times Series Australia\2021 Census TSP Statistical Area 2 for AUS'
t01 = pd.read_csv(f"{tsp_path}\\2021Census_T01_AUST_SA2.csv")
print(f"\nTotal columns: {t01.columns}")


Total columns: Index(['SA2_CODE_2021', 'Tot_persons_C11_M', 'Tot_persons_C11_F',
       'Tot_persons_C11_P', 'Tot_persons_C16_M', 'Tot_persons_C16_F',
       'Tot_persons_C16_P', 'Tot_persons_C21_M', 'Tot_persons_C21_F',
       'Tot_persons_C21_P',
       ...
       'LSH_Oth_Lan_2021_Ce_P', 'Aust_citiz_C11_M', 'Aust_citiz_C11_F',
       'Aust_citiz_C11_P', 'Aust_citiz_C16_M', 'Aust_citiz_C16_F',
       'Aust_citiz_C16_P', 'Aust_citiz_C21_M', 'Aust_citiz_C21_F',
       'Aust_citiz_C21_P'],
      dtype='str', length=199)


In [18]:
# from above columns we need only population columns of 2016 and 2021
tsp_path = r'D:\full AUS\Times Series Australia\2021 Census TSP Statistical Area 2 for AUS'
t01 = pd.read_csv(
    f"{tsp_path}\\2021Census_T01_AUST_SA2.csv",
    usecols=['SA2_CODE_2021', 'Tot_persons_C16_P', 'Tot_persons_C21_P']
)

# Friendly rename
t01 = t01.rename(columns={
    'SA2_CODE_2021': 'sa2_code',
    'Tot_persons_C16_P': 'pop_2016',
    'Tot_persons_C21_P': 'pop_2021'
})
# now we need to calculate population growth between 2016 and 2021 
t01['pop_growth_pct']=np.where(t01['pop_2016']>0,
                               ((t01['pop_2021']-t01['pop_2016'])/t01['pop_2016'])*100,
                               np.nan)

In [19]:
print(f"Rows: {len(t01)}")
print(f"SA2s with zero 2016 population (NaN growth): {(t01['pop_2016'] == 0).sum()}\n")

print("Population growth % stats:")
print(t01['pop_growth_pct'].describe())

# Sanity checks — fastest growing and shrinking SA2s
print("\nTop 5 fastest-growing SA2s (expect new housing estates):")
print(t01.nlargest(5, 'pop_growth_pct')[['sa2_code', 'pop_2016', 'pop_2021', 'pop_growth_pct']])

print("\nTop 5 fastest-shrinking SA2s:")
print(t01.nsmallest(5, 'pop_growth_pct')[['sa2_code', 'pop_2016', 'pop_2021', 'pop_growth_pct']])

Rows: 2472
SA2s with zero 2016 population (NaN growth): 45

Population growth % stats:
count    2427.000000
mean       23.461142
std       221.102389
min      -100.000000
25%         1.565783
50%         4.893661
75%         9.955753
max      5157.425743
Name: pop_growth_pct, dtype: float64

Top 5 fastest-growing SA2s (expect new housing estates):
       sa2_code  pop_2016  pop_2021  pop_growth_pct
2382  801041120       101      5310     5157.425743
1008  212031556       299     14802     4850.501672
2358  801011143        16       714     4362.500000
1092  213051584       178      7197     3943.258427
1090  213051582       288     10132     3418.055556

Top 5 fastest-shrinking SA2s:
       sa2_code  pop_2016  pop_2021  pop_growth_pct
642   197979799       419         0          -100.0
778   206041127         6         0          -100.0
783   206041507         4         0          -100.0
1959  504031064         3         0          -100.0
2025  506021120         6         0          -1

In [20]:
import pandas as pd
import numpy as np

# t01 already has pop_growth_pct computed correctly
# Decision: cap extreme growth at 99th percentile to prevent tiny-base outliers
# from distorting linear models (tree models are robust either way)
#we didn't cap by our assumption we asked model to decide and cap at whatever value at top 1% begins.
# if value more tham 199.4 it sets them to 199.4 else it doen't change it
p99 = t01['pop_growth_pct'].quantile(0.99)
print(f"99th percentile of growth: {p99:.1f}%")
print(f"SA2s above this cap: {(t01['pop_growth_pct'] > p99).sum()}\n")

# Apply the cap
t01['pop_growth_pct_capped'] = t01['pop_growth_pct'].clip(upper=p99)

# Compare before/after
print("Before cap:")
print(t01['pop_growth_pct'].describe())
print("\nAfter cap:")
print(t01['pop_growth_pct_capped'].describe())

# Use the capped version as our feature
t01_clean = t01[['sa2_code', 'pop_growth_pct_capped']].rename(
    columns={'pop_growth_pct_capped': 'pop_growth_pct'}
)

# If we observe that the std changes from 221 to 28, that means we can see how much those 25 outliers inflated.we did this from distorting linear models


99th percentile of growth: 199.4%
SA2s above this cap: 25

Before cap:
count    2427.000000
mean       23.461142
std       221.102389
min      -100.000000
25%         1.565783
50%         4.893661
75%         9.955753
max      5157.425743
Name: pop_growth_pct, dtype: float64

After cap:
count    2427.000000
mean       10.135648
std        28.671215
min      -100.000000
25%         1.565783
50%         4.893661
75%         9.955753
max       199.446337
Name: pop_growth_pct_capped, dtype: float64


In [21]:
# merge this into our male file
features=pd.read_csv('national_features_v1.csv')
features = features.merge(t01_clean, on='sa2_code', how='outer')
features.to_csv('national_features_v1b.csv', index=False)

**Note:** This section reads the ABS SEIFA 2021 SA2 Excel file (publicly downloadable from [ABS SEIFA 2021](https://www.abs.gov.au/statistics/people/people-and-communities/socio-economic-indexes-areas-seifa-australia)). The file isn't included in this repo (data source belongs to ABS). Update the path below to your local copy if re-running. The computed SEIFA features are already saved in `national_features_v3.csv`, so subsequent cells run without this section.

In [22]:
# now we are checking for SEIFA values where we are focusing mostly on IER beacuse it captures income,wealth(housing,rent,mortgage)
import pandas as pd

seifa_path = r'D:\Statistical Area Level 2, Indexes, SEIFA 2021.xlsx'

# List all worksheet (tab) names in the Excel file
xls = pd.ExcelFile(seifa_path)
print("Worksheets in the SEIFA file:")
for i, sheet in enumerate(xls.sheet_names):
    print(f"  {i}: {sheet}")

Worksheets in the SEIFA file:
  0: Contents
  1: Table 1
  2: Table 2
  3: Table 3
  4: Table 4
  5: Table 5
  6: Table 6
  7: Explanatory Notes


In [23]:
peek=pd.read_excel(seifa_path,sheet_name='Table 1',header=None, nrows=10)
print(peek.to_string())


                                                           0                                          1                                              2       3                                                            4       5                            6       7                                  8       9                          10
0                             Australian Bureau of Statistics                                        NaN                                            NaN     NaN                                                          NaN     NaN                          NaN     NaN                                NaN     NaN                        NaN
1          Socio-Economic Indexes for Australia (SEIFA), 2021                                        NaN                                            NaN     NaN                                                          NaN     NaN                          NaN     NaN                                NaN     NaN                    

In [24]:
# we can see data starts from row 6
seifa = pd.read_excel(seifa_path, sheet_name='Table 1', skiprows=6, header=None)
# The columns are in a known fixed order (0-indexed):
#  0: SA2 code        1: SA2 name
#  2: IRSD score      3: IRSD decile
#  4: IRSAD score     5: IRSAD decile
#  6: IER score       7: IER decile
#  8: IEO score       9: IEO decile
#  10: population
# Keep only the columns we want (code + 8 SEIFA features) and name them.
seifa = seifa.iloc[:, [0, 2, 3, 4, 5, 6, 7, 8, 9]]
seifa.columns = [
    'sa2_code',
    'seifa_irsd_score',  'seifa_irsd_decile',
    'seifa_irsad_score', 'seifa_irsad_decile',
    'seifa_ier_score',   'seifa_ier_decile',
    'seifa_ieo_score',   'seifa_ieo_decile',
]

print(f"Shape: {seifa.shape}")
print(f"\nFirst 4 rows:")
print(seifa.head(4).to_string())
print(f"\nDtypes:")
print(seifa.dtypes)

Shape: (2368, 9)

First 4 rows:
    sa2_code seifa_irsd_score seifa_irsd_decile seifa_irsad_score seifa_irsad_decile seifa_ier_score seifa_ier_decile  seifa_ieo_score  seifa_ieo_decile
0  101021007             1024                 6              1001                  6            1027                7           1008.0               6.0
1  101021008              994                 5               982                  5            1000                5            967.0               5.0
2  101021009             1010                 5               998                  6             945                3           1000.0               6.0
3  101021010             1025                 6              1015                  6             969                4           1025.0               7.0

Dtypes:
sa2_code               object
seifa_irsd_score       object
seifa_irsd_decile      object
seifa_irsad_score      object
seifa_irsad_decile     object
seifa_ier_score        object
seifa_ier_deci

In [25]:
#now we need to convert them to numeric if any values failconverts them into Nan
cols_to_check = [
    'seifa_irsd_score', 'seifa_irsd_decile',
    'seifa_irsad_score', 'seifa_irsad_decile',
    'seifa_ier_score', 'seifa_ier_decile'
]
for col in cols_to_check:
    # Try converting to numeric; find which values FAIL
    numeric=pd.to_numeric(seifa[col], errors='coerce')
    failed=seifa[numeric.isna() & seifa[col].notna()]
    if len(failed) > 0:
        print(f"{col}: {len(failed)} non-numeric value(s)")
        print(f"   Examples: {failed[col].unique()[:5]}")
    else:
        print(f"{col}: all numeric (object dtype is just from mixed reading)")

seifa_irsd_score: 13 non-numeric value(s)
   Examples: ['-']
seifa_irsd_decile: 13 non-numeric value(s)
   Examples: ['-']
seifa_irsad_score: 13 non-numeric value(s)
   Examples: ['-']
seifa_irsad_decile: 13 non-numeric value(s)
   Examples: ['-']
seifa_ier_score: 12 non-numeric value(s)
   Examples: ['-']
seifa_ier_decile: 12 non-numeric value(s)
   Examples: ['-']


In [26]:
#Convert everything to numeric; '-' suppression markers become NaN 
for col in seifa.columns:
    seifa[col] = pd.to_numeric(seifa[col], errors='coerce')

# --- 4. Drop footnote/junk rows (where sa2_code isn't a valid number) and fix dtype ---
seifa = seifa.dropna(subset=['sa2_code'])
seifa['sa2_code'] = seifa['sa2_code'].astype('int64')

# --- 5. Verify ---
print(f"Shape: {seifa.shape}")
print(f"\nDtypes:")
print(seifa.dtypes)

Shape: (2366, 9)

Dtypes:
sa2_code                int64
seifa_irsd_score      float64
seifa_irsd_decile     float64
seifa_irsad_score     float64
seifa_irsad_decile    float64
seifa_ier_score       float64
seifa_ier_decile      float64
seifa_ieo_score       float64
seifa_ieo_decile      float64
dtype: object


In [27]:
import pandas as pd

# Reload the CLEAN 5-feature checkpoint fresh from disk
features = pd.read_csv('national_features_v1b.csv')
print(f"Checkpoint columns ({list(features.columns)}")
print(f"Checkpoint shape: {features.shape}\n")

# Safety check: confirm v1b does NOT already contain SEIFA columns
seifa_already_present = [c for c in features.columns if 'seifa' in c.lower()]
if seifa_already_present:
    print(f"WARNING: checkpoint already has SEIFA columns: {seifa_already_present}")
    print("This means v1b was saved wrong. Stop and tell me.")
else:
    print("Checkpoint is clean (5 features, no SEIFA). Safe to merge.\n")
    
    # Merge SEIFA exactly once
    features = features.merge(seifa, on='sa2_code', how='outer')
    
    print(f"After merge: {features.shape}")
    print(f"Columns ({len(features.columns)}):")
    for c in features.columns:
        print(f"  {c}")
    
    print(f"\nMissing values:")
    print(features.isna().sum())
    
    # Save v2
    features.to_csv('national_features_v2.csv', index=False)

Checkpoint columns (['sa2_code', 'population', 'median_mortgage_monthly', 'construction_pct', 'retail_pct', 'pop_growth_pct']
Checkpoint shape: (2472, 6)

Checkpoint is clean (5 features, no SEIFA). Safe to merge.

After merge: (2472, 14)
Columns (14):
  sa2_code
  population
  median_mortgage_monthly
  construction_pct
  retail_pct
  pop_growth_pct
  seifa_irsd_score
  seifa_irsd_decile
  seifa_irsad_score
  seifa_irsad_decile
  seifa_ier_score
  seifa_ier_decile
  seifa_ieo_score
  seifa_ieo_decile

Missing values:
sa2_code                     0
population                   0
median_mortgage_monthly    112
construction_pct           126
retail_pct                 126
pop_growth_pct              45
seifa_irsd_score           119
seifa_irsd_decile          119
seifa_irsad_score          119
seifa_irsad_decile         119
seifa_ier_score            118
seifa_ier_decile           118
seifa_ieo_score            106
seifa_ieo_decile           106
dtype: int64


In [28]:
import geopandas as gpd
import pandas as pd
shp_path = r'D:\SA2_2021_AUST_SHP_GDA2020\SA2_2021_AUST_GDA2020.shp'
sa2 = gpd.read_file(shp_path)

print(f"SA2 regions loaded: {len(sa2)}")
print(f"CRS (coordinate system): {sa2.crs}")
print(f"Columns: {list(sa2.columns)}\n")
#project degrees (EPSG:7844) -> metres (EPSG:3577, Australian Albers, equal-area)
sa2_metres = sa2.to_crs(epsg=3577)
centroids_metres = sa2_metres.geometry.centroid
# convert centroid points back to lat/lon
centroids_latlon = centroids_metres.to_crs(epsg=4326)
# Build a clean lookup table
sa2_centroids = pd.DataFrame({
    'sa2_code': sa2['SA2_CODE21'].astype('int64'),
    'sa2_name': sa2['SA2_NAME21'],
    'centroid_lat': centroids_latlon.y,
    'centroid_lon': centroids_latlon.x
})

print("First 5 SA2 centroids:")
print(sa2_centroids.head().to_string())

SA2 regions loaded: 2473
CRS (coordinate system): EPSG:7844
Columns: ['SA2_CODE21', 'SA2_NAME21', 'CHG_FLAG21', 'CHG_LBL21', 'SA3_CODE21', 'SA3_NAME21', 'SA4_CODE21', 'SA4_NAME21', 'GCC_CODE21', 'GCC_NAME21', 'STE_CODE21', 'STE_NAME21', 'AUS_CODE21', 'AUS_NAME21', 'AREASQKM21', 'LOCI_URI21', 'geometry']



ValueError: invalid literal for int() with base 10: 'ZZZZZZZZZ'

In [29]:
# removing rows whose code isn't all digits and also removes zzz rows and fixes the error
valid = sa2['SA2_CODE21'].astype(str).str.isdigit()
print(f"Non-numeric SA2 codes removed: {(~valid).sum()}")
print(f"Examples of removed codes: {sa2.loc[~valid, 'SA2_CODE21'].unique()[:5]}")
sa2 = sa2[valid].copy()
print(f"SA2 regions after cleaning: {len(sa2)}\n")
empty_geom = sa2.geometry.isna() | sa2.geometry.is_empty
if empty_geom.any():
    print(f"Empty geometries removed: {empty_geom.sum()}")
    sa2 = sa2[~empty_geom].copy()
    print(f"SA2 regions after geometry cleaning: {len(sa2)}\n")


sa2_metres = sa2.to_crs(epsg=3577)
centroids_metres = sa2_metres.geometry.centroid
centroids_latlon = centroids_metres.to_crs(epsg=4326)
# Build clean lookup table
sa2_centroids = pd.DataFrame({
    'sa2_code': sa2['SA2_CODE21'].astype('int64').values,
    'sa2_name': sa2['SA2_NAME21'].values,
    'centroid_lat': centroids_latlon.y.values,
    'centroid_lon': centroids_latlon.x.values
})

print("First 5 SA2 centroids:")
print(sa2_centroids.head().to_string())
sa2_centroids.to_csv('sa2_centroids.csv', index=False)


Non-numeric SA2 codes removed: 1
Examples of removed codes: <StringArray>
['ZZZZZZZZZ']
Length: 1, dtype: str
SA2 regions after cleaning: 2472

Empty geometries removed: 18
SA2 regions after geometry cleaning: 2454

First 5 SA2 centroids:
    sa2_code                         sa2_name  centroid_lat  centroid_lon
0  101021007                        Braidwood    -35.454377    149.793814
1  101021008                          Karabar    -35.375953    149.232796
2  101021009                       Queanbeyan    -35.351043    149.225457
3  101021010                Queanbeyan - East    -35.355170    149.252407
4  101021012  Queanbeyan West - Jerrabomberra    -35.377591    149.202844


In [30]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd

# Load the national centroids and competitors
centroids = pd.read_csv('sa2_centroids.csv')
comps = pd.read_csv('competitors_national.csv')

# Inline test — same as Phase 0a, verify the formula gives a sensible answer
lat1, lon1 = radians(-34.9285), radians(138.6007)
lat2, lon2 = radians(-34.9163), radians(138.6232)
dlat = lat2 - lat1
dlon = lon2 - lon1
a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
test = 6371 * 2 * atan2(sqrt(a), sqrt(1-a))
print(f"Inline test: {test:.2f} km (should be ~2.4 km — two points in central Adelaide)\n")

results = []
for _, sa2 in centroids.iterrows():
    sa2_code = sa2['sa2_code']
    lat1 = radians(sa2['centroid_lat'])
    lon1 = radians(sa2['centroid_lon'])

    comp_distances = []
    for _, comp in comps.iterrows():
        lat2 = radians(comp['latitude'])
        lon2 = radians(comp['longitude'])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        a = min(a, 1.0)
        dist = 6371 * 2 * atan2(sqrt(a), sqrt(1-a))
        comp_distances.append(dist)

    results.append({
        'sa2_code': sa2_code,
        'dist_nearest_competitor_km': min(comp_distances),
        'competitors_within_5km': sum(1 for d in comp_distances if d <= 5),
        'competitors_within_10km': sum(1 for d in comp_distances if d <= 10),
    })

dist_df = pd.DataFrame(results)
dist_df.to_csv(r'D:\full AUS\sa2_distance_features.csv', index=False)

Inline test: 2.46 km (should be ~2.4 km — two points in central Adelaide)



In [31]:
#merge both files and checking if there is already this file to avoid _x/_y duplication of columns
features = pd.read_csv('national_features_v2.csv')
dist = pd.read_csv('sa2_distance_features.csv')
already = [c for c in features.columns if 'competitor' in c.lower()]
if already:
    print(f"Distance columns already present: {already} — reload v2 first!")
else:
    features = features.merge(dist, on='sa2_code', how='outer')
    features.to_csv('national_features_v3.csv', index=False)

In [32]:
features = pd.read_csv('national_features_v3.csv')
bunnings = pd.read_csv('bunnings_complete_with_sa2.csv')
positive_sa2s = set(bunnings['sa2_code'].unique())
POP_THRESHOLD = 3000
before = len(features)
features = features[features['population'] >= POP_THRESHOLD].copy()
print(f"\nAfter population >= {POP_THRESHOLD} filter: {before} -> {len(features)} SA2s")

# --- Add the Bunnings target column ---
features['has_bunnings'] = features['sa2_code'].isin(positive_sa2s).astype(int)
print(features.isna().sum())


After population >= 3000 filter: 2472 -> 2231 SA2s
sa2_code                       0
population                     0
median_mortgage_monthly       10
construction_pct               0
retail_pct                     0
pop_growth_pct                 0
seifa_irsd_score               6
seifa_irsd_decile              6
seifa_irsad_score              6
seifa_irsad_decile             6
seifa_ier_score                6
seifa_ier_decile               6
seifa_ieo_score                6
seifa_ieo_decile               6
dist_nearest_competitor_km     6
competitors_within_5km         6
competitors_within_10km        6
has_bunnings                   0
dtype: int64


In [33]:
#replacing missing values with the column medain value
feature_cols = [
    'population', 'median_mortgage_monthly', 'construction_pct', 'retail_pct',
    'pop_growth_pct', 'seifa_irsd_score', 'seifa_irsd_decile',
    'seifa_irsad_score', 'seifa_irsad_decile', 'seifa_ier_score', 'seifa_ier_decile',
    'seifa_ieo_score', 'seifa_ieo_decile', 'dist_nearest_competitor_km',
    'competitors_within_5km', 'competitors_within_10km'
]
for col in feature_cols:
    median_val = features[col].median()
    features[col] = features[col].fillna(median_val)

print(f"Missing values AFTER imputation:  {features[feature_cols].isna().sum().sum()} total")
features.to_csv('national_model_ready.csv', index=False)

Missing values AFTER imputation:  0 total


In [34]:
adl=pd.read_csv(r"D:\adelaide-bunnings-expansion\17_features_Adl\adelaide_master_v2.csv")
# How many Adelaide SA2s, and the positive/negative split?
print(f"Total Adelaide SA2s: {len(adl)}")
print(f"With Bunnings (positive): {(adl['bunnings_count'] > 0).sum()}")
print(f"Without Bunnings (negative): {(adl['bunnings_count'] == 0).sum()}")

# Extract the set of Adelaide SA2 codes
adelaide_sa2 = set(adl['SA2_CODE_2021'].unique())
print(f"\nUnique Adelaide SA2 codes: {len(adelaide_sa2)}")
print(f"Sample codes: {sorted(adelaide_sa2)[:5]}")

Total Adelaide SA2s: 112
With Bunnings (positive): 16
Without Bunnings (negative): 96

Unique Adelaide SA2 codes: 112
Sample codes: [np.int64(401011001), np.int64(401011002), np.int64(401021003), np.int64(401021004), np.int64(401021005)]


In [35]:
# now we need to assign train and test(Adelaide)
features = pd.read_csv(r'D:\full AUS\national_model_ready.csv')
test=features[features['sa2_code'].isin(adelaide_sa2)].copy()
train=features[~features['sa2_code'].isin(adelaide_sa2)].copy()
# Sanity: confirm no Adelaide SA2 leaked into training
overlap = set(train['sa2_code']) & adelaide_sa2
print(len(overlap))

0


In [36]:
# now we need to train with differnt models to find best model
from sklearn. preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
#StratifiedKFold is tool that splits data into balanced folds for crossvalidation. cross_val_score is function that actually runs model through those folds and return scores.
from sklearn.pipeline import make_pipeline
#this chains steps together(scale,then model) into one object and scaling happen inside cross-validation safely
from xgboost import XGBClassifier
features = pd.read_csv(r'D:\full AUS\national_model_ready.csv')
all_features = [
    'population', 'median_mortgage_monthly', 'construction_pct', 'retail_pct',
    'pop_growth_pct', 'seifa_irsd_score', 'seifa_irsd_decile',
    'seifa_irsad_score', 'seifa_irsad_decile', 'seifa_ier_score', 'seifa_ier_decile',
    'seifa_ieo_score', 'seifa_ieo_decile', 'dist_nearest_competitor_km',
    'competitors_within_5km', 'competitors_within_10km'
]
#splits train into x,y as x=16features and y=just(has_bunnings has 0/1 values) feature here model learns relationship between x andy
x,y=train[all_features],train['has_bunnings']
# it's the random baseline for PR-AUC
base_rate=y.mean()
print(base_rate)
#creates fold-spitter and stores in cv. n_splits=5 means 5folds(train on 4/5, test on 1/5 and rotate so, each split will be on test set.
#shuffle=true randomizes which rows go into so, folds are not biased
cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'Logistic Regression (L1)': make_pipeline(StandardScaler(),
        LogisticRegression(penalty='l1', C=0.5, solver='liblinear',
                           class_weight='balanced', max_iter=1000, random_state=42)),
    'Logistic Regression (L2)': make_pipeline(StandardScaler(),
        LogisticRegression(penalty='l2', C=1.0, class_weight='balanced',
                           max_iter=1000, random_state=42)),
    'Random Forest': RandomForestClassifier(n_estimators=500, max_depth=3,
        min_samples_leaf=5, class_weight='balanced', random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=500, max_depth=3,
        min_samples_leaf=5, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=2,
        learning_rate=0.05, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.05,
        scale_pos_weight=6, random_state=42, eval_metric='logloss'),
    'SVM (RBF)': make_pipeline(StandardScaler(),
        SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)),
    'KNN': make_pipeline(StandardScaler(),
        KNeighborsClassifier(n_neighbors=15)),
}

results={}
#model.items gives both name and model object each iteration
for name, model in models.items():
    #cross_val_score(---) does heavy lifting it takes model runs the full 5-fold routine using cv splitter, scoring each fold roc-auc and returns an array of all five values and scoring='average_precision that's sklearn name for PR-AUC
    roc=cross_val_score(model, x, y, cv=cv, scoring='roc_auc')
    pr=cross_val_score(model, x, y, cv=cv, scoring='average_precision')
    results[name]=(roc.mean(), roc.std(), pr.mean(), pr.std())
for name, (rm,rs,pm,ps) in sorted(results.items(), key= lambda x: -x[1][0]):
    print(f"{name:<26} {rm:.3f}, {rs:.3f}, {pm:.3f}, {ps:.3f}")
best_roc = max(results.items(), key=lambda x: x[1][0])
best_pr  = max(results.items(), key=lambda x: x[1][2])
print(f"\n  Best ROC-AUC: {best_roc[0]} ({best_roc[1][0]:.3f})")
print(f"  Best PR-AUC:  {best_pr[0]} ({best_pr[1][2]:.3f})")
    

0.13104744011272898


C:\Users\kiran\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\kiran\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\kiran\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty=

Random Forest              0.745, 0.019, 0.334, 0.033
Gradient Boosting          0.743, 0.020, 0.332, 0.016
SVM (RBF)                  0.740, 0.015, 0.316, 0.019
Logistic Regression (L1)   0.740, 0.014, 0.308, 0.047
Logistic Regression (L2)   0.739, 0.013, 0.305, 0.045
XGBoost                    0.737, 0.011, 0.361, 0.027
Extra Trees                0.731, 0.028, 0.303, 0.037
KNN                        0.698, 0.021, 0.278, 0.028

  Best ROC-AUC: Random Forest (0.745)
  Best PR-AUC:  XGBoost (0.361)


In [37]:
# fine tune XGBoost with gridsearchcv
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier
xgb_grid = {
    'n_estimators': [100, 200, 400],        # number of trees
    'max_depth': [2, 3, 4],                  # how deep each tree can go
    'learning_rate': [0.03, 0.05, 0.1],      # how much each tree contributes
    'scale_pos_weight': [4, 6, 8],           # imbalance weighting (around our 6)
}

xgb_base = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_search=GridSearchCV(estimator=xgb_base,
                        param_grid=xgb_grid, # chose ranges around xgb_grid settings
                        scoring='average_precision',
                        cv=cv,
                        n_jobs=-1, #use all cpu cores
                        verbose=1) #prints progess we can see it's working
xgb_search.fit(x, y)
print(f"Best PR-AUC: {xgb_search.best_score_:.3f}")
print(f"Best params: {xgb_search.best_params_}")

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best PR-AUC: 0.367
Best params: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 200, 'scale_pos_weight': 4}


In [38]:
xgb=XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.05,
                    scale_pos_weight=4, random_state=42, eval_metric='logloss')
xgb.fit(x, y)
importance = pd.DataFrame({
    'feature': all_features,
    'importance_pct': xgb.feature_importances_*100
}).sort_values('importance_pct', ascending=False).reset_index(drop=True)
for _,r in importance.iterrows():
     print(f"  {r['feature']:28s} {r['importance_pct']:0.1f}%")
    

  population                   22.4%
  retail_pct                   10.6%
  median_mortgage_monthly      9.9%
  seifa_irsad_score            9.0%
  dist_nearest_competitor_km   8.2%
  seifa_ier_score              7.7%
  seifa_irsd_decile            5.9%
  competitors_within_10km      5.8%
  construction_pct             4.7%
  seifa_ieo_score              4.7%
  pop_growth_pct               4.0%
  seifa_irsd_score             4.0%
  competitors_within_5km       3.0%
  seifa_irsad_decile           0.0%
  seifa_ier_decile             0.0%
  seifa_ieo_decile             0.0%


In [39]:
# got features percentages train rest of national and predict on adelaide
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
features = pd.read_csv(r'D:\full AUS\national_model_ready.csv')
all_features = [
    'population', 'median_mortgage_monthly', 'construction_pct', 'retail_pct',
    'pop_growth_pct', 'seifa_irsd_score', 'seifa_irsd_decile',
    'seifa_irsad_score', 'seifa_irsad_decile', 'seifa_ier_score', 'seifa_ier_decile',
    'seifa_ieo_score', 'seifa_ieo_decile', 'dist_nearest_competitor_km',
    'competitors_within_5km', 'competitors_within_10km'
]
train = features[~features['sa2_code'].isin(adelaide_sa2)].copy()
test = features[features['sa2_code'].isin(adelaide_sa2)].copy()
x_train, y_train = train[all_features], train['has_bunnings']
x_test, y_test = test[all_features], test['has_bunnings']
# train tuned XGBoost model
test= test.copy()
test['score']=xgb.predict_proba(x_test)[:,1]
test['rank']=test['score'].rank(ascending=False).astype(int)
# AUC and PR-AUC on adelaide
auc=roc_auc_score(y_test, test['score'])
pr= average_precision_score(y_test, test['score'])
print(f"  ROC-AUC: {auc:.3f}")
print(f"  PR-AUC:  {pr:.3f} ")
# did it rank adelaide real bunnnings highly?
actual=test[test['has_bunnings']==1].sort_values('score', ascending=False)
n_test=len(test)
print(actual[['sa2_code', 'score', 'rank']].to_string(index=False))

top_quartile=int(n_test/4)
in_tq=(actual['rank']<=top_quartile).sum()
print(f"How many real bunnings ranked in top quartlie out of 13 positives: {in_tq} ")
# precision at top-k
#it checks top 10,15,20,25  recommendations and checks and adds to hits which has bunnings
# and makes it a percentage ex: top 10 = 50% means 5/10 has real bunnings that means model's ten most-confiedent adelaide picks genuinely hasve bunnings.
for k in [10,15,20,25]:
    topk=test.nlargest(k,'score')
    hits=topk['has_bunnings'].sum()
    print(f" Top{k:2d}: {hits}/{k} are real bunnings ({hits/k*100:.0f}% precision)")


top15 =test.nlargest(15, 'score')[['sa2_code','score', 'has_bunnings','dist_nearest_competitor_km']]
print(top15.to_string(index=False))

  ROC-AUC: 0.805
  PR-AUC:  0.430 
 sa2_code    score  rank
402011026 0.823083     2
401021007 0.811195     3
402031038 0.785352     4
403041177 0.773127     6
403021064 0.769002     8
402021032 0.706428    13
402051052 0.649009    20
404011097 0.643830    22
401061021 0.601170    32
403041086 0.569522    41
403021058 0.561810    42
404031108 0.523619    49
401051017 0.247890    75
How many real bunnings ranked in top quartlie out of 13 positives: 8 
 Top10: 5/10 are real bunnings (50% precision)
 Top15: 6/15 are real bunnings (40% precision)
 Top20: 7/20 are real bunnings (35% precision)
 Top25: 8/25 are real bunnings (32% precision)
 sa2_code    score  has_bunnings  dist_nearest_competitor_km
402021031 0.824443             0                    1.635156
402011026 0.823083             1                    1.627049
401021007 0.811195             1                    0.438235
402031038 0.785352             1                    0.973013
402021028 0.776729             0                    

In [40]:
# let's add suburb name into top 15 recommendations
# Drop suburb if it already exists from a previous run (avoids the duplicate-column error)
test = test.drop(columns=['suburb'], errors='ignore')
#names from centroids file
names=pd.read_csv(r'D:\full AUS\sa2_centroids.csv')[['sa2_code','sa2_name']]
names=names.rename(columns={'sa2_name' : 'suburb'})
#merge into test
test=test.merge(names, on='sa2_code', how ='left')
test['rank']=test['score'].rank(ascending=False, method='first')
top15 = test.nlargest(15, 'score')[[
    'suburb', 'sa2_code', 'score', 'rank', 'has_bunnings', 'dist_nearest_competitor_km'
]]
print(top15.to_string(index=False))

                      suburb  sa2_code    score  rank  has_bunnings  dist_nearest_competitor_km
              Elizabeth East 402021031 0.824443   1.0             0                    1.635156
              Gawler - South 402011026 0.823083   2.0             1                    1.627049
                Mount Barker 401021007 0.811195   3.0             1                    0.438235
             Windsor Gardens 402031038 0.785352   4.0             1                    0.973013
       Craigmore - Blakeview 402021028 0.776729   5.0             0                    5.048235
   Seaford - Seaford Meadows 403041177 0.773127   6.0             1                    0.452276
               Morphettville 403021062 0.770411   7.0             0                    1.524064
                   Warradale 403021064 0.769002   8.0             1                    0.932760
                    Adelaide 401011001 0.755872   9.0             0                    2.786382
                Davoren Park 402021029 0